# Itchy — TTT Meta-Learning Validation

Tests whether meta-learning teaches LoRA adapters to adapt per-document.

**Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**

In [ ]:
!pip install -q torch numpy huggingface-hub
import torch
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

In [ ]:
# Download data and tokenizer
import os, shutil, numpy as np
from pathlib import Path
from huggingface_hub import hf_hub_download

os.makedirs('data', exist_ok=True)

def download(filename, subfolder, dest):
    if Path(dest).exists():
        print(f'  {dest} already exists')
        return
    print(f'  Downloading {filename} from {subfolder}...')
    cached = hf_hub_download(
        repo_id='willdepueoai/parameter-golf',
        filename=filename,
        subfolder=subfolder,
        repo_type='dataset',
    )
    cached = str(Path(cached).resolve())
    try:
        os.link(cached, dest)
    except OSError:
        shutil.copy2(cached, dest)
    print(f'  -> {dest} ({Path(dest).stat().st_size / 1e6:.1f}MB)')

# Data shards
for name in ['fineweb_train_000000.bin', 'fineweb_val_000000.bin']:
    download(name, 'datasets/datasets/fineweb10B_sp1024', f'data/{name}')

# Tokenizer
download('fineweb_1024_bpe.model', 'datasets/tokenizers', 'data/fineweb_1024_bpe.model')
download('fineweb_1024_bpe.vocab', 'datasets/tokenizers', 'data/fineweb_1024_bpe.vocab')

# Verify
for f in ['data/fineweb_train_000000.bin', 'data/fineweb_val_000000.bin', 'data/fineweb_1024_bpe.model']:
    assert Path(f).exists(), f'{f} missing!'
print('\nAll files downloaded OK')

In [ ]:
# Convert sp1024 tokens to raw bytes
import sentencepiece as spm

def load_token_shard(path):
    header_bytes = 256 * np.dtype('<i4').itemsize
    header = np.fromfile(path, dtype='<i4', count=256)
    num_tokens = int(header[2])
    return np.fromfile(path, dtype='<u2', count=num_tokens, offset=header_bytes)

def write_byte_shard(path, byte_values):
    header = np.zeros(256, dtype='<i4')
    header[0] = 20240520
    header[1] = 1
    header[2] = len(byte_values)
    with open(path, 'wb') as f:
        f.write(header.tobytes())
        f.write(byte_values.astype('<u2').tobytes())

try:
    sp = spm.SentencePieceProcessor(model_file='data/fineweb_1024_bpe.model')
except:
    !pip install -q sentencepiece
    import sentencepiece as spm
    sp = spm.SentencePieceProcessor(model_file='data/fineweb_1024_bpe.model')

os.makedirs('data/bytes', exist_ok=True)
for name in ['fineweb_train_000000.bin', 'fineweb_val_000000.bin']:
    out_path = f'data/bytes/{name}'
    if Path(out_path).exists():
        print(f'{name} already converted')
        continue
    print(f'Converting {name} to bytes...')
    tokens = load_token_shard(f'data/{name}')
    all_bytes = []
    chunk_size = 100_000
    for start in range(0, len(tokens), chunk_size):
        text = sp.decode(tokens[start:start+chunk_size].tolist())
        all_bytes.append(np.frombuffer(text.encode('utf-8'), dtype=np.uint8))
    byte_data = np.concatenate(all_bytes)
    write_byte_shard(out_path, byte_data)
    print(f'  {len(tokens):,} tokens -> {len(byte_data):,} bytes')

print('Byte data ready')

In [ ]:
# ============================================================
# MODEL DEFINITION (PyTorch version of Itchy)
# ============================================================
import math
import time
import torch
import torch.nn.functional as F
from torch import Tensor, nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


class LoRAAdapter(nn.Module):
    def __init__(self, dim, rank):
        super().__init__()
        self.down = nn.Parameter(torch.randn(rank, dim) / math.sqrt(dim))
        self.up = nn.Parameter(torch.zeros(dim, rank))
        self.gate = nn.Parameter(torch.tensor(0.0))

    def forward(self, x):
        return self.gate * (x @ self.down.to(x.dtype).T @ self.up.to(x.dtype).T)


class Rotary(nn.Module):
    def __init__(self, dim, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        self.register_buffer('inv_freq', inv_freq, persistent=False)
        self._cache = None

    def forward(self, seq_len, device, dtype):
        if self._cache is None or self._cache[0] != seq_len:
            t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
            freqs = torch.outer(t, self.inv_freq.to(device))
            self._cache = (seq_len, freqs.cos()[None,None,:,:], freqs.sin()[None,None,:,:])
        return self._cache[1].to(dtype), self._cache[2].to(dtype)


def apply_rotary_emb(x, cos, sin):
    h = x.size(-1) // 2
    x1, x2 = x[..., :h], x[..., h:]
    return torch.cat((x1*cos + x2*sin, x1*(-sin) + x2*cos), dim=-1)


class Attention(nn.Module):
    def __init__(self, dim, num_heads, num_kv_heads, rope_base, qk_gain):
        super().__init__()
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = dim // num_heads
        kv_dim = num_kv_heads * self.head_dim
        self.c_q = nn.Linear(dim, dim, bias=False)
        self.c_k = nn.Linear(dim, kv_dim, bias=False)
        self.c_v = nn.Linear(dim, kv_dim, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)
        nn.init.zeros_(self.proj.weight)
        self.q_gain = nn.Parameter(torch.full((num_heads,), qk_gain))
        self.rotary = Rotary(self.head_dim, rope_base)

    def forward(self, x):
        B, S, D = x.shape
        q = self.c_q(x).reshape(B, S, self.num_heads, self.head_dim).transpose(1,2)
        k = self.c_k(x).reshape(B, S, self.num_kv_heads, self.head_dim).transpose(1,2)
        v = self.c_v(x).reshape(B, S, self.num_kv_heads, self.head_dim).transpose(1,2)
        q = F.rms_norm(q, (q.size(-1),))
        k = F.rms_norm(k, (k.size(-1),))
        cos, sin = self.rotary(S, x.device, q.dtype)
        q = apply_rotary_emb(q, cos, sin) * self.q_gain.to(q.dtype)[None,:,None,None]
        k = apply_rotary_emb(k, cos, sin)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                          enable_gqa=(self.num_kv_heads != self.num_heads))
        return self.proj(y.transpose(1,2).reshape(B, S, D))


class MLP(nn.Module):
    def __init__(self, dim, mult):
        super().__init__()
        self.fc = nn.Linear(dim, dim*mult, bias=False)
        self.proj = nn.Linear(dim*mult, dim, bias=False)
        nn.init.zeros_(self.proj.weight)

    def forward(self, x):
        return self.proj(torch.relu(self.fc(x)).square())


class Block(nn.Module):
    def __init__(self, dim, num_heads, num_kv_heads, mlp_mult, rope_base, qk_gain, adapter_rank):
        super().__init__()
        self.attn = Attention(dim, num_heads, num_kv_heads, rope_base, qk_gain)
        self.mlp = MLP(dim, mlp_mult)
        self.attn_scale = nn.Parameter(torch.ones(dim))
        self.mlp_scale = nn.Parameter(torch.ones(dim))
        self.adapter = LoRAAdapter(dim, adapter_rank)

    def forward(self, x, x0):
        attn_out = self.attn(F.rms_norm(x, (x.size(-1),)))
        attn_out = attn_out + self.adapter(attn_out)
        x = x + self.attn_scale.to(x.dtype)[None,None,:] * attn_out
        x = x + self.mlp_scale.to(x.dtype)[None,None,:] * self.mlp(F.rms_norm(x, (x.size(-1),)))
        return x


class Itchy(nn.Module):
    def __init__(self, dim, num_layers, num_heads, num_kv_heads, mlp_mult,
                 patch_size, adapter_rank, softcap=30.0, rope_base=10000.0, qk_gain=1.5):
        super().__init__()
        self.dim = dim
        self.patch_size = patch_size
        self.softcap = softcap
        self.byte_embed = nn.Embedding(260, dim)
        self.patch_proj = nn.Linear(dim * patch_size, dim, bias=False)
        self.blocks = nn.ModuleList([
            Block(dim, num_heads, num_kv_heads, mlp_mult, rope_base, qk_gain, adapter_rank)
            for _ in range(num_layers)
        ])
        self.final_norm = nn.Identity()  # rms_norm applied inline
        self.unpatch = nn.Linear(dim, patch_size * 260, bias=False)
        self.num_enc = num_layers // 2
        self.num_dec = num_layers - self.num_enc
        self.skip_weights = nn.Parameter(torch.ones(min(self.num_enc, self.num_dec), dim))

    def forward_features(self, byte_ids):
        B, S = byte_ids.shape
        x = self.byte_embed(byte_ids)
        x = x.reshape(B, S // self.patch_size, self.patch_size * self.dim)
        x = F.rms_norm(self.patch_proj(x), (self.dim,))
        x0 = x
        skips = []
        for i in range(self.num_enc):
            x = self.blocks[i](x, x0)
            skips.append(x)
        for i in range(self.num_dec):
            if skips:
                x = x + self.skip_weights[i].to(x.dtype)[None,None,:] * skips.pop()
            x = self.blocks[self.num_enc + i](x, x0)
        return F.rms_norm(x, (x.size(-1),))

    def forward(self, byte_ids, targets):
        x = self.forward_features(byte_ids)
        logits = self.unpatch(x).reshape(-1, 260)
        logits = self.softcap * torch.tanh(logits / self.softcap)
        return F.cross_entropy(logits.float(), targets.reshape(-1))

    def adapter_params(self):
        return [p for n, p in self.named_parameters() if 'adapter' in n]

    def get_adapter_state(self):
        return [(b.adapter.down.data.clone(), b.adapter.up.data.clone(), b.adapter.gate.data.clone())
                for b in self.blocks]

    def set_adapter_state(self, state):
        with torch.no_grad():
            for b, (d, u, g) in zip(self.blocks, state):
                b.adapter.down.data.copy_(d)
                b.adapter.up.data.copy_(u)
                b.adapter.gate.data.copy_(g)

    def reset_adapters(self):
        with torch.no_grad():
            for b in self.blocks:
                b.adapter.gate.data.zero_()
                b.adapter.up.data.zero_()


n_params = lambda m: sum(p.numel() for p in m.parameters())
print('Model defined')

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================

def load_byte_shard(path):
    header_bytes = 256 * np.dtype('<i4').itemsize
    header = np.fromfile(path, dtype='<i4', count=256)
    num_tokens = int(header[2])
    return np.fromfile(path, dtype='<u2', count=num_tokens, offset=header_bytes).astype(np.int64)

train_data = load_byte_shard('data/bytes/fineweb_train_000000.bin')
val_data = load_byte_shard('data/bytes/fineweb_val_000000.bin')
print(f'Train: {len(train_data):,} bytes | Val: {len(val_data):,} bytes')

In [ ]:
# ============================================================
# CONFIG
# ============================================================

# Model (small enough for T4 16GB, big enough to learn)
DIM = 256
NUM_LAYERS = 8
NUM_HEADS = 8
NUM_KV_HEADS = 4
MLP_MULT = 2
PATCH_SIZE = 4
ADAPTER_RANK = 16
SEQ_LEN = 512

# Training
TRAIN_STEPS = 3000
BATCH_SIZE = 8192  # bytes per batch
LR = 3e-4
LOG_EVERY = 200

# TTT meta-learning
TTT_RATIO = 0.2        # 20% of steps are meta-learning
TTT_INNER_STEPS = 3
TTT_INNER_LR = 0.01

# TTT evaluation test
TEST_DOCS = 50
TEST_TTT_STEPS = 5
TEST_TTT_LR = 0.005

print(f'Config: dim={DIM} layers={NUM_LAYERS} patch={PATCH_SIZE} rank={ADAPTER_RANK}')

In [ ]:
# ============================================================
# PHASE 1: TRAIN WITH TTT META-LEARNING
# ============================================================

torch.manual_seed(42)
model = Itchy(
    dim=DIM, num_layers=NUM_LAYERS, num_heads=NUM_HEADS,
    num_kv_heads=NUM_KV_HEADS, mlp_mult=MLP_MULT,
    patch_size=PATCH_SIZE, adapter_rank=ADAPTER_RANK,
).to(device).bfloat16()

print(f'Model: {n_params(model):,} params ({n_params(model)/1e6:.1f}M)')
print(f'Adapter params: {sum(p.numel() for p in model.adapter_params()):,}')

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=0.01)
scaler = torch.amp.GradScaler()

ttt_interval = max(int(1.0 / TTT_RATIO), 1)
pos = 0
t0 = time.time()
losses_regular = []
losses_ttt = []

print(f'\nTraining {TRAIN_STEPS} steps (TTT every {ttt_interval} steps)...')
print(f'Expected time: ~10-20 min on T4\n')

for step in range(1, TRAIN_STEPS + 1):
    # Get batch
    usable = (BATCH_SIZE // SEQ_LEN) * SEQ_LEN
    end = pos + usable + 1
    if end > len(train_data):
        pos = 0
        end = usable + 1
    chunk = train_data[pos:end]
    pos = end - 1
    x = torch.tensor(chunk[:-1].reshape(-1, SEQ_LEN), device=device)
    y = torch.tensor(chunk[1:].reshape(-1, SEQ_LEN), device=device)

    is_ttt = step > 0 and step % ttt_interval == 0

    if is_ttt:
        # TTT META-LEARNING EPISODE
        half = SEQ_LEN // 2
        ctx_x, ctx_y = x[:1, :half], y[:1, :half]
        tgt_x, tgt_y = x[:1, half:], y[:1, half:]

        # Save adapter state
        adapter_snap = model.get_adapter_state()

        # Inner loop: SGD on adapters using context
        adapter_ps = model.adapter_params()
        for _ in range(TTT_INNER_STEPS):
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                ctx_loss = model(ctx_x, ctx_y)
            grads = torch.autograd.grad(ctx_loss, adapter_ps, create_graph=False)
            with torch.no_grad():
                for p, g in zip(adapter_ps, grads):
                    p.data -= TTT_INNER_LR * g

        # Outer loss on target — gradients flow to ALL params
        optimizer.zero_grad()
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            outer_loss = model(tgt_x, tgt_y)
        outer_loss.backward()
        optimizer.step()

        # Restore adapters
        model.set_adapter_state(adapter_snap)
        losses_ttt.append(outer_loss.item())
    else:
        # REGULAR TRAINING STEP
        optimizer.zero_grad()
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            loss = model(x, y)
        loss.backward()
        optimizer.step()
        losses_regular.append(loss.item())

    if step % LOG_EVERY == 0 or step == 1:
        avg_reg = np.mean(losses_regular[-LOG_EVERY:]) if losses_regular else 0
        avg_ttt = np.mean(losses_ttt[-LOG_EVERY//ttt_interval:]) if losses_ttt else 0
        bpb = avg_reg / math.log(2)
        elapsed = time.time() - t0
        print(f'step {step:>5} | loss {avg_reg:.4f} | bpb {bpb:.4f} | '
              f'ttt_loss {avg_ttt:.4f} | {elapsed:.0f}s')

print(f'\nTraining done in {time.time()-t0:.0f}s')

In [ ]:
# ============================================================
# PHASE 2: TEST — DOES ADAPTER ADAPTATION HELP?
# ============================================================

print('='*70)
print('PHASE 2: Testing TTT adaptation on validation documents')
print('='*70)
print(f'{TEST_DOCS} docs | {TEST_TTT_STEPS} adapt steps | lr={TEST_TTT_LR}')
print()

doc_len = SEQ_LEN * 2
half = SEQ_LEN

results_none = []
results_adapter = []
results_allparam = []

# Clear any cached RoPE tensors from inference mode
for b in model.blocks:
    b.attn.rotary._cache = None

for doc_idx in range(TEST_DOCS):
    start = doc_idx * doc_len * 3
    if start + doc_len + 1 > len(val_data):
        break
    chunk = val_data[start:start + doc_len + 1]
    ctx_x = torch.tensor(chunk[:half].reshape(1,-1), device=device)
    ctx_y = torch.tensor(chunk[1:half+1].reshape(1,-1), device=device)
    tgt_x = torch.tensor(chunk[half:half+half].reshape(1,-1), device=device)
    tgt_y = torch.tensor(chunk[half+1:half+half+1].reshape(1,-1), device=device)

    # A) No adaptation (use no_grad instead of inference_mode to avoid poisoning cache)
    with torch.no_grad():
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            loss_none = model(tgt_x, tgt_y).item()
    results_none.append(loss_none)

    # B) Adapter-only adaptation
    adapter_snap = model.get_adapter_state()
    adapter_ps = model.adapter_params()
    for _ in range(TEST_TTT_STEPS):
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            ctx_loss = model(ctx_x, ctx_y)
        grads = torch.autograd.grad(ctx_loss, adapter_ps)
        with torch.no_grad():
            for p, g in zip(adapter_ps, grads):
                p.data -= TEST_TTT_LR * g
    with torch.no_grad():
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            loss_adapter = model(tgt_x, tgt_y).item()
    results_adapter.append(loss_adapter)
    model.set_adapter_state(adapter_snap)

    # C) All-param adaptation
    full_snap = {k: v.clone() for k, v in model.state_dict().items()}
    all_ps = list(model.parameters())
    for _ in range(TEST_TTT_STEPS):
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            ctx_loss = model(ctx_x, ctx_y)
        grads = torch.autograd.grad(ctx_loss, all_ps)
        with torch.no_grad():
            for p, g in zip(all_ps, grads):
                p.data -= TEST_TTT_LR * g
    with torch.no_grad():
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            loss_allparam = model(tgt_x, tgt_y).item()
    results_allparam.append(loss_allparam)
    model.load_state_dict(full_snap)

    if (doc_idx + 1) % 10 == 0:
        a = np.mean(results_none[-10:])
        b = np.mean(results_adapter[-10:])
        c = np.mean(results_allparam[-10:])
        print(f'  docs {doc_idx-8:>2}-{doc_idx+1:>2} | '
              f'none: {a:.4f} | adapter: {b:.4f} ({b-a:+.4f}) | '
              f'all: {c:.4f} ({c-a:+.4f})')

In [ ]:
# ============================================================
# FINAL RESULTS
# ============================================================

print()
print('='*70)
print('FINAL RESULTS')
print('='*70)
a = np.mean(results_none)
b = np.mean(results_adapter)
c = np.mean(results_allparam)
print(f'  No adaptation:     {a:.4f} nats  ({a/math.log(2):.4f} BPB)')
print(f'  Adapter TTT:       {b:.4f} nats  ({b/math.log(2):.4f} BPB)  delta: {b-a:+.4f}')
print(f'  All-param TTT:     {c:.4f} nats  ({c/math.log(2):.4f} BPB)  delta: {c-a:+.4f}')
print()

n = len(results_none)
n_b = sum(1 for x,y in zip(results_none, results_adapter) if y < x)
n_c = sum(1 for x,y in zip(results_none, results_allparam) if y < x)
print(f'  Adapter TTT helped:   {n_b}/{n} docs ({100*n_b/n:.0f}%)')
print(f'  All-param TTT helped: {n_c}/{n} docs ({100*n_c/n:.0f}%)')
print()

if b - a < -0.01:
    print('  >>> ADAPTER TTT WORKS — meta-learning taught adapters to adapt!')
    print('  >>> This validates the Itchy thesis.')
elif b - a < 0:
    print('  >>> Adapter TTT shows weak improvement — needs more training or tuning.')
else:
    print('  >>> Adapter TTT did not help — meta-learning may need more steps/epochs.')

if c - a < -0.01:
    print('  >>> All-param TTT works — per-document adaptation signal confirmed.')

In [ ]:
# ============================================================
# PHASE 3: HEAD-TO-HEAD — Itchy byte-level vs Token-level baseline
# Same params (~4.3M), same data, same steps, same GPU
# ============================================================

print('='*70)
print('PHASE 3: Head-to-head comparison')
print('='*70)

# --- Token-level baseline model (same architecture, 1024 vocab) ---

class BaselineGPT(nn.Module):
    """Simplified baseline: token-level, 1024 vocab, tied embeddings."""
    def __init__(self, vocab_size, dim, num_layers, num_heads, num_kv_heads, mlp_mult,
                 softcap=30.0, rope_base=10000.0, qk_gain=1.5):
        super().__init__()
        self.dim = dim
        self.softcap = softcap
        self.tok_emb = nn.Embedding(vocab_size, dim)
        nn.init.normal_(self.tok_emb.weight, std=0.005)
        self.blocks = nn.ModuleList([
            Block(dim, num_heads, num_kv_heads, mlp_mult, rope_base, qk_gain, adapter_rank=0)
            for _ in range(num_layers)
        ])
        # Remove adapters (rank=0 would fail, so we just won't use them)
        for b in self.blocks:
            b.adapter = nn.Identity()
        self.num_enc = num_layers // 2
        self.num_dec = num_layers - self.num_enc
        self.skip_weights = nn.Parameter(torch.ones(min(self.num_enc, self.num_dec), dim))

    def forward(self, input_ids, targets):
        x = F.rms_norm(self.tok_emb(input_ids), (self.dim,))
        x0 = x
        skips = []
        for i in range(self.num_enc):
            x = self.blocks[i](x, x0)
            skips.append(x)
        for i in range(self.num_dec):
            if skips:
                x = x + self.skip_weights[i].to(x.dtype)[None,None,:] * skips.pop()
            x = self.blocks[self.num_enc + i](x, x0)
        x = F.rms_norm(x, (x.size(-1),))
        # Tied embedding head
        logits = F.linear(x, self.tok_emb.weight)
        logits = self.softcap * torch.tanh(logits / self.softcap)
        return F.cross_entropy(logits.float().reshape(-1, 1024), targets.reshape(-1))

# We need token-level data too
print('Loading token-level data...')
token_train = load_byte_shard('data/fineweb_train_000000.bin')  # these are actually tokens, not bytes
token_val = load_byte_shard('data/fineweb_val_000000.bin')
print(f'Token train: {len(token_train):,} | Token val: {len(token_val):,}')

# Match param count: find dim that gives ~4.3M params with 1024 vocab
# 1024 vocab embedding = 1024*dim params (big!)
# Itchy had dim=256, 8 layers, ~4.3M params
# With 1024 vocab at dim=128: embed=131K, but model capacity much smaller
# Let's try dim=128, 8 layers
BASELINE_DIM = 128
BASELINE_LAYERS = 8

# Need to fix Block to work without adapter
class SimpleBlock(nn.Module):
    def __init__(self, dim, num_heads, num_kv_heads, mlp_mult, rope_base, qk_gain):
        super().__init__()
        self.attn = Attention(dim, num_heads, num_kv_heads, rope_base, qk_gain)
        self.mlp = MLP(dim, mlp_mult)
        self.attn_scale = nn.Parameter(torch.ones(dim))
        self.mlp_scale = nn.Parameter(torch.ones(dim))

    def forward(self, x, x0):
        attn_out = self.attn(F.rms_norm(x, (x.size(-1),)))
        x = x + self.attn_scale.to(x.dtype)[None,None,:] * attn_out
        x = x + self.mlp_scale.to(x.dtype)[None,None,:] * self.mlp(F.rms_norm(x, (x.size(-1),)))
        return x

class BaselineGPTv2(nn.Module):
    def __init__(self, vocab_size, dim, num_layers, num_heads, num_kv_heads, mlp_mult,
                 softcap=30.0, rope_base=10000.0, qk_gain=1.5):
        super().__init__()
        self.dim = dim
        self.softcap = softcap
        self.tok_emb = nn.Embedding(vocab_size, dim)
        nn.init.normal_(self.tok_emb.weight, std=0.005)
        self.blocks = nn.ModuleList([
            SimpleBlock(dim, num_heads, num_kv_heads, mlp_mult, rope_base, qk_gain)
            for _ in range(num_layers)
        ])
        self.num_enc = num_layers // 2
        self.num_dec = num_layers - self.num_enc
        self.skip_weights = nn.Parameter(torch.ones(min(self.num_enc, self.num_dec), dim))

    def forward(self, input_ids, targets):
        x = F.rms_norm(self.tok_emb(input_ids), (self.dim,))
        x0 = x
        skips = []
        for i in range(self.num_enc):
            x = self.blocks[i](x, x0)
            skips.append(x)
        for i in range(self.num_dec):
            if skips:
                x = x + self.skip_weights[i].to(x.dtype)[None,None,:] * skips.pop()
            x = self.blocks[self.num_enc + i](x, x0)
        x = F.rms_norm(x, (x.size(-1),))
        logits = F.linear(x, self.tok_emb.weight)
        logits = self.softcap * torch.tanh(logits / self.softcap)
        return F.cross_entropy(logits.float().reshape(-1, 1024), targets.reshape(-1))

torch.manual_seed(42)
baseline = BaselineGPTv2(
    vocab_size=1024, dim=BASELINE_DIM, num_layers=BASELINE_LAYERS,
    num_heads=4, num_kv_heads=2, mlp_mult=2,
).to(device).bfloat16()

baseline_params = sum(p.numel() for p in baseline.parameters())
print(f'\nBaseline: {baseline_params:,} params ({baseline_params/1e6:.1f}M)')
print(f'Itchy:    {n_params(model):,} params ({n_params(model)/1e6:.1f}M)')

# Train baseline same number of steps
baseline_opt = torch.optim.AdamW(baseline.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=0.01)
BASELINE_SEQ = 1024  # token-level uses longer sequences
BASELINE_BATCH = BATCH_SIZE  # same byte budget

pos = 0
t0 = time.time()
baseline_losses = []

print(f'\nTraining baseline for {TRAIN_STEPS} steps...\n')
for step in range(1, TRAIN_STEPS + 1):
    usable = (BASELINE_BATCH // BASELINE_SEQ) * BASELINE_SEQ
    if usable == 0:
        usable = BASELINE_SEQ
    end = pos + usable + 1
    if end > len(token_train):
        pos = 0
        end = usable + 1
    chunk = token_train[pos:end]
    pos = end - 1
    x = torch.tensor(chunk[:-1].reshape(-1, BASELINE_SEQ), device=device)
    y = torch.tensor(chunk[1:].reshape(-1, BASELINE_SEQ), device=device)

    baseline_opt.zero_grad()
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        loss = baseline(x, y)
    loss.backward()
    baseline_opt.step()
    baseline_losses.append(loss.item())

    if step % LOG_EVERY == 0 or step == 1:
        avg = np.mean(baseline_losses[-LOG_EVERY:])
        elapsed = time.time() - t0
        print(f'  step {step:>5} | loss {avg:.4f} | {elapsed:.0f}s')

print(f'Baseline training done in {time.time()-t0:.0f}s')

# Now evaluate BOTH on validation data
print('\n' + '='*70)
print('VALIDATION COMPARISON')
print('='*70)

# Itchy val (byte-level): BPB = loss / ln(2)
itchy_val_losses = []
for i in range(0, min(100_000, len(val_data) - SEQ_LEN - 1), SEQ_LEN):
    chunk = val_data[i:i + SEQ_LEN + 1]
    x = torch.tensor(chunk[:-1].reshape(1, -1), device=device)
    y = torch.tensor(chunk[1:].reshape(1, -1), device=device)
    with torch.no_grad():
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            loss = model(x, y).item()
    itchy_val_losses.append(loss)
itchy_val = np.mean(itchy_val_losses)
itchy_bpb = itchy_val / math.log(2)

# Baseline val (token-level): BPB needs token-to-byte conversion
# Approximate: avg ~2.4 bytes per token for sp1024
baseline_val_losses = []
for i in range(0, min(50_000, len(token_val) - BASELINE_SEQ - 1), BASELINE_SEQ):
    chunk = token_val[i:i + BASELINE_SEQ + 1]
    x = torch.tensor(chunk[:-1].reshape(1, -1), device=device)
    y = torch.tensor(chunk[1:].reshape(1, -1), device=device)
    with torch.no_grad():
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            loss = baseline(x, y).item()
    baseline_val_losses.append(loss)
baseline_val = np.mean(baseline_val_losses)
# BPB for token model: bits_per_token * tokens_per_byte
# bits_per_token = loss / ln(2), tokens_per_byte ≈ 1/2.44
bytes_per_token = 2.44  # approximate for sp1024
baseline_bpb = (baseline_val / math.log(2)) / bytes_per_token

print(f'\n  Itchy (byte-level):')
print(f'    val_loss: {itchy_val:.4f} nats')
print(f'    val_bpb:  {itchy_bpb:.4f}')
print(f'    params:   {n_params(model):,}')
print(f'\n  Baseline (token-level):')
print(f'    val_loss: {baseline_val:.4f} nats')
print(f'    val_bpb:  {baseline_bpb:.4f} (approx, assumes {bytes_per_token:.2f} bytes/token)')
print(f'    params:   {baseline_params:,}')
print(f'\n  Delta BPB: {itchy_bpb - baseline_bpb:+.4f} (negative = Itchy wins)')